# 🌌 APEX Launcher

Welcome to the **APEX** runtime environment. This notebook is a minimal loader that delegates all execution, dependencies, and synchronization to the `APEX Bootstrap Manager`.

In [ ]:
import sys
import subprocess
from pathlib import Path

# =========================================
# APEX Launcher Configuration
# =========================================
LAUNCHER_CONFIG = {
    "repository": "https://github.com/Nikhil-Kumar-Shah/APEX.git",
    "branch": "main",
    "target": "/content/APEX",
    "auto_update": True,
    "mount_drive": True,
}

# 1. Environment Detection & Storage
is_colab = "google.colab" in sys.modules
if is_colab:
    print("[INFO] Google Colab detected.")
    if LAUNCHER_CONFIG["mount_drive"]:
        try:
            from google.colab import drive
            drive.mount("/content/drive")
            print("[SUCCESS] Drive Mounted.")
        except Exception:
            print("[WARNING] Drive mount failed. Using temporary storage.")

# 2. Stage-0 Clone (Solves the Bootstrapping Paradox)
target_dir = Path(LAUNCHER_CONFIG["target"]) if is_colab else Path.cwd().parent / "APEX"
if not (target_dir / ".git").exists():
    print("[INFO] Fetching APEX Runtime...")
    target_dir.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(["git", "clone", "-b", LAUNCHER_CONFIG["branch"], LAUNCHER_CONFIG["repository"], str(target_dir)], check=True)

if str(target_dir.resolve()) not in sys.path:
    sys.path.insert(0, str(target_dir.resolve()))

# 3. Stage-1 Handover (BootstrapManager)
from bootstrap.manager import RuntimeContext, BootstrapManager

context = RuntimeContext(
    environment="colab" if is_colab else "local",
    persistence=Path("/content/drive/MyDrive/APEX") if is_colab else target_dir.parent / "APEX_Data",
    runtime=target_dir
)

print("[INFO] Starting APEX... Logging will now redirect to the Runtime Console.")
BootstrapManager(context).launch()
